# India Tractor Industry — Forecasting Notebook
## FY2026-27 Sales Forecast by HP Segment using DuckDB + Time-Series Models

**Pipeline:** Raw Data → Databricks Medallion (Bronze/Silver/Gold) → DuckDB → statsmodels → Forecast

| Item | Detail |
|------|--------|
| Data Source | TMA Annual Reports, IMD (Rainfall), FRED (PPI) |
| Methods | Holt-Winters Exponential Smoothing, ARIMA, DuckDB SQL Moving Average |
| Forecast Horizon | FY2025-26, FY2026-27 |
| Output | Total sales forecast + HP segment breakdown + Emerging states |

---

## 0. Setup — Install Dependencies

In [ ]:
# Run this cell first in Google Colab
!pip install -q duckdb statsmodels pmdarima plotly pandas numpy scikit-learn

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from matplotlib.gridspec import GridSpec
import duckdb
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_percentage_error

plt.style.use('dark_background')
GREEN  = '#10b981'
AMBER  = '#f59e0b'
BLUE   = '#3b82f6'
PURPLE = '#8b5cf6'
RED    = '#ef4444'
GRAY   = '#6b7280'

print('All dependencies loaded successfully.')

## 1. Data Loading

Data sourced from:
- **TMA (Tractor Manufacturers Association)** — Annual sales figures
- **IMD** — Southwest Monsoon rainfall (Jun-Sep)
- **FRED** — Farm Equipment PPI (US proxy for input cost inflation)
- **ICRA / TMA** — HP segment estimates

If running from Databricks Gold layer exports, replace the inline data with:
```python
india = pd.read_csv('gold_market_summary.csv')  # from Databricks
```

In [ ]:
# ── India Annual Tractor Sales (TMA) ──────────────────────────────────────
india = pd.DataFrame({
    'fiscal_year': [
        '2003-04','2004-05','2005-06','2006-07','2007-08','2008-09',
        '2009-10','2010-11','2011-12','2012-13','2013-14','2014-15',
        '2015-16','2016-17','2017-18','2018-19','2019-20','2020-21',
        '2021-22','2022-23','2023-24','2024-25',
    ],
    'year_start': list(range(2003, 2025)),
    'units_sold': [
        276000, 302000, 346000, 363000, 367000, 317000,
        432000, 520000, 562000, 543000, 567000, 510000,
        521000, 582000, 693000, 874000, 735000, 891000,
        939000, 932000, 918000, 912000,
    ]
})
india['units_lakh'] = india['units_sold'] / 100_000
india['yoy_pct']    = india['units_sold'].pct_change() * 100

# ── Southwest Monsoon Rainfall (IMD, Jun-Sep, mm) ─────────────────────────
rainfall = pd.DataFrame({
    'year_start':  list(range(2010, 2025)),
    'rainfall_mm': [916, 901, 860, 966, 764, 761, 863, 887, 905, 881, 958, 901, 925, 820, 934],
    'normal_mm':   [868]*15,
})
rainfall['deficit'] = rainfall['rainfall_mm'] < rainfall['normal_mm']

# ── HP Segment Split (ICRA estimates) ─────────────────────────────────────
hp_data = pd.DataFrame({
    'fiscal_year': ['2018-19','2019-20','2020-21','2021-22','2022-23','2023-24','2024-25'],
    'year_start':  [2018, 2019, 2020, 2021, 2022, 2023, 2024],
    'below_30hp':  [44100, 36700, 44500, 46900, 46600, 45900, 45600],
    'hp_30_40':    [392000, 330700, 400900, 422000, 418800, 412000, 409400],
    'hp_41_50':    [258800, 217900, 263700, 278000, 275900, 271700, 269900],
    'hp_51_60':    [105700,  88900, 107600, 113600, 112700, 110900, 110200],
    'above_60hp':  [ 72400,  61000,  73300,  78500,  78000,  76500,  76900],
})
hp_data['total'] = hp_data[['below_30hp','hp_30_40','hp_41_50','hp_51_60','above_60hp']].sum(axis=1)

print(f'India data: {len(india)} years ({india.fiscal_year.iloc[0]} to {india.fiscal_year.iloc[-1]})')
print(f'HP data:    {len(hp_data)} years')
print(f'Rainfall:   {len(rainfall)} years')
print(f'Latest sales: {india.units_sold.iloc[-1]:,} units ({india.units_lakh.iloc[-1]:.2f}L)')

## 2. Exploratory Data Analysis

In [ ]:
fig = plt.figure(figsize=(18, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Sales trend
ax1 = fig.add_subplot(gs[0, :])
ax1.fill_between(india.fiscal_year, india.units_lakh, alpha=0.15, color=GREEN)
ax1.plot(india.fiscal_year, india.units_lakh, color=GREEN, lw=2.5, marker='o', ms=4)
ax1.set_title('India Annual Tractor Sales FY2003-2025', fontsize=13)
ax1.set_ylabel('Units (Lakh)')
ax1.tick_params(axis='x', rotation=45)
for yr, lbl, val in [('2008-09','GFC',3.17), ('2019-20','NBFC',7.35), ('2020-21','+21%',8.91)]:
    ax1.annotate(lbl, xy=(yr, val), xytext=(yr, val + 0.4),
                 ha='center', fontsize=8, color=AMBER,
                 arrowprops=dict(arrowstyle='->', color=AMBER, lw=1.2))

# 2. YoY growth
ax2 = fig.add_subplot(gs[1, 0])
colors = [GREEN if v >= 0 else RED for v in india.yoy_pct.dropna()]
ax2.bar(india.fiscal_year[1:], india.yoy_pct.dropna(), color=colors, width=0.7)
ax2.axhline(0, color='white', lw=0.8, ls='--')
ax2.set_title('YoY Growth (%)', fontsize=11)
ax2.tick_params(axis='x', rotation=90)
ax2.yaxis.set_major_formatter(mtick.PercentFormatter())

# 3. Rainfall overlay (2010 onward)
rain_m = pd.merge(india[india.year_start >= 2010], rainfall, on='year_start')
ax3 = fig.add_subplot(gs[1, 1])
ax3b = ax3.twinx()
ax3.scatter(rain_m.fiscal_year, rain_m.units_lakh,
            c=[RED if d else GREEN for d in rain_m.deficit], s=60, zorder=3)
ax3b.plot(rain_m.fiscal_year, rain_m.rainfall_mm, color=BLUE, lw=1.5, alpha=0.7)
ax3.set_title('Sales vs Rainfall', fontsize=11)
ax3.set_ylabel('Sales (Lakh)', fontsize=9)
ax3b.set_ylabel('Rainfall (mm)', fontsize=9)
ax3.tick_params(axis='x', rotation=90)

# 4. HP segment bar
ax4 = fig.add_subplot(gs[1, 2])
segs = ['below_30hp','hp_30_40','hp_41_50','hp_51_60','above_60hp']
labels = ['<30HP','30-40HP','41-50HP','51-60HP','>60HP']
clrs   = [GRAY, GREEN, AMBER, BLUE, PURPLE]
last   = hp_data.iloc[-1]
shares = [last[s] / last['total'] * 100 for s in segs]
bars   = ax4.bar(labels, shares, color=clrs, width=0.65)
for bar, val in zip(bars, shares):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=8)
ax4.set_title('HP Segment Mix FY2024-25', fontsize=11)
ax4.set_ylabel('Share (%)')

plt.suptitle('India Tractor Market — EDA Summary', fontsize=15, fontweight='bold', y=1.01)
plt.savefig('eda_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA chart saved as eda_summary.png')

## 3. DuckDB SQL-based Time-Series Analysis

DuckDB lets us run SQL window functions directly on DataFrames — no database server required.
This mirrors what would run on the Databricks Gold layer exports.

In [ ]:
con = duckdb.connect()

# Register DataFrames as virtual tables
con.register('india_sales',   india)
con.register('hp_segments',   hp_data)
con.register('india_rainfall', rainfall)

# ── Query 1: Rolling averages and YoY delta ───────────────────────────────
q1 = """
SELECT
    fiscal_year,
    year_start,
    units_sold,
    ROUND(AVG(units_sold) OVER (
        ORDER BY year_start
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    )) AS ma3_units,
    ROUND(AVG(units_sold) OVER (
        ORDER BY year_start
        ROWS BETWEEN 4 PRECEDING AND CURRENT ROW
    )) AS ma5_units,
    units_sold - LAG(units_sold, 1) OVER (ORDER BY year_start) AS yoy_delta,
    ROUND(
        (units_sold - LAG(units_sold,1) OVER (ORDER BY year_start))
        / LAG(units_sold,1) OVER (ORDER BY year_start) * 100, 2
    ) AS yoy_pct
FROM india_sales
ORDER BY year_start
"""
df_trend = con.execute(q1).df()
print('Rolling averages:')
print(df_trend[['fiscal_year','units_sold','ma3_units','ma5_units','yoy_delta','yoy_pct']].tail(10).to_string(index=False))

In [ ]:
# ── Query 2: Forecast using MA3 + avg recent growth ───────────────────────
q2 = """
WITH base AS (
    SELECT year_start, units_sold,
           AVG(units_sold) OVER (
               ORDER BY year_start
               ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
           ) AS ma3,
           units_sold - LAG(units_sold,1) OVER (ORDER BY year_start) AS delta
    FROM india_sales
),
recent_trend AS (
    SELECT
        AVG(delta)              AS avg_delta,
        STDDEV(delta)           AS stddev_delta,
        MAX(ma3)                AS latest_ma3,
        MAX(units_sold)         AS latest_actual
    FROM base
    WHERE year_start >= 2020
)
SELECT
    '2025-26'                                                         AS forecast_fy,
    ROUND(latest_actual + avg_delta)                                  AS delta_forecast,
    ROUND(latest_ma3    * 1.018)                                      AS ma3_forecast,
    ROUND(latest_actual + avg_delta - stddev_delta)                   AS lower_bound,
    ROUND(latest_actual + avg_delta + stddev_delta)                   AS upper_bound,
    ROUND(avg_delta)                                                  AS avg_annual_delta,
    ROUND(stddev_delta)                                               AS stddev_annual_delta
FROM recent_trend
"""
df_fc_duckdb = con.execute(q2).df()
print('DuckDB SQL Forecast for FY2025-26:')
print(df_fc_duckdb.to_string(index=False))

In [ ]:
# ── Query 3: HP Segment weighted-share forecast ───────────────────────────
q3 = """
WITH hp_long AS (
    SELECT year_start, 'Below 30 HP' AS segment, below_30hp AS units FROM hp_segments
    UNION ALL
    SELECT year_start, '30-40 HP',   hp_30_40    FROM hp_segments
    UNION ALL
    SELECT year_start, '41-50 HP',   hp_41_50    FROM hp_segments
    UNION ALL
    SELECT year_start, '51-60 HP',   hp_51_60    FROM hp_segments
    UNION ALL
    SELECT year_start, 'Above 60 HP', above_60hp  FROM hp_segments
),
totals AS (
    SELECT year_start, SUM(units) AS total_units FROM hp_long GROUP BY year_start
),
shares AS (
    SELECT h.year_start, h.segment,
           ROUND(h.units * 100.0 / t.total_units, 3) AS share_pct
    FROM hp_long h JOIN totals t USING(year_start)
),
weighted AS (
    SELECT segment,
           -- Recent 3 years weighted 2x for recency bias
           ROUND(
               SUM(CASE WHEN year_start >= 2022 THEN share_pct * 2 ELSE share_pct END)
               / SUM(CASE WHEN year_start >= 2022 THEN 2.0 ELSE 1.0 END), 3
           ) AS weighted_share
    FROM shares GROUP BY segment
)
SELECT
    segment,
    weighted_share                       AS forecast_share_pct,
    ROUND(940000 * weighted_share / 100) AS fy2526_units,
    ROUND(960000 * weighted_share / 100) AS fy2627_units
FROM weighted
ORDER BY forecast_share_pct DESC
"""
df_hp_fc = con.execute(q3).df()
print('HP Segment Forecast (DuckDB):')
print(df_hp_fc.to_string(index=False))

## 4. Stationarity Test (ADF)

In [ ]:
ts = india.set_index('year_start')['units_sold']

def adf_test(series, name='Series'):
    result = adfuller(series.dropna(), autolag='AIC')
    print(f'\n--- ADF Test: {name} ---')
    print(f'  ADF Statistic : {result[0]:.4f}')
    print(f'  p-value       : {result[1]:.4f}')
    print(f'  Critical (5%) : {result[4]["5%"]:.4f}')
    print(f'  Stationary    : {"YES" if result[1] < 0.05 else "NO — differencing needed"}')

adf_test(ts,            'Raw Sales')
adf_test(ts.diff(),     'First Difference')
adf_test(ts.pct_change(), 'YoY Growth Rate')

## 5. Holt-Winters Exponential Smoothing

In [ ]:
TRAIN_CUTOFF = 2021
FORECAST_PERIODS = 2  # FY2025-26 and FY2026-27

train_ts = india[india.year_start <= TRAIN_CUTOFF]['units_sold']
test_ts  = india[india.year_start >  TRAIN_CUTOFF]['units_sold']

# Additive trend, no seasonality (annual data)
hw_model = ExponentialSmoothing(
    train_ts,
    trend='add',
    seasonal=None,
    initialization_method='estimated'
).fit(optimized=True)

hw_fitted   = hw_model.fittedvalues
hw_forecast = hw_model.forecast(FORECAST_PERIODS + len(test_ts))

# Split forecast
hw_fc_test = hw_forecast.iloc[:len(test_ts)]
hw_fc_fwd  = hw_forecast.iloc[len(test_ts):]

# Evaluation on test set
mape_hw = mean_absolute_percentage_error(test_ts, hw_fc_test[:len(test_ts)])

print(f'Holt-Winters Parameters:')
print(f'  Alpha (level)  : {hw_model.params["smoothing_level"]:.4f}')
print(f'  Beta  (trend)  : {hw_model.params["smoothing_trend"]:.4f}')
print(f'  MAPE on test   : {mape_hw:.2%}')

fc_labels = ['2025-26', '2026-27'][:FORECAST_PERIODS]
for lbl, val in zip(fc_labels, hw_fc_fwd):
    print(f'  Forecast FY{lbl}: {val:,.0f} units ({val/100000:.2f}L)')

## 6. ARIMA Model

In [ ]:
# Try auto ARIMA if pmdarima is available, otherwise use manual ARIMA(1,1,1)
try:
    from pmdarima import auto_arima
    arima_model = auto_arima(
        train_ts, start_p=0, start_q=0, max_p=3, max_q=3, d=1,
        seasonal=False, stepwise=True, suppress_warnings=True, information_criterion='aic'
    )
    order = arima_model.order
    print(f'Auto ARIMA selected: ARIMA{order}')
    arima_fc = arima_model.predict(n_periods=FORECAST_PERIODS + len(test_ts))

except ImportError:
    print('pmdarima not found — using ARIMA(1,1,1) via statsmodels')
    arima_fit = SARIMAX(train_ts, order=(1,1,1), trend='c').fit(disp=False)
    arima_fc  = arima_fit.forecast(steps=FORECAST_PERIODS + len(test_ts))
    print(arima_fit.summary().tables[0])

arima_fc_test = arima_fc[:len(test_ts)]
arima_fc_fwd  = arima_fc[len(test_ts):]

mape_arima = mean_absolute_percentage_error(test_ts, arima_fc_test)
print(f'\nARIMA MAPE on test set: {mape_arima:.2%}')

for lbl, val in zip(fc_labels, arima_fc_fwd):
    print(f'  Forecast FY{lbl}: {val:,.0f} units ({val/100000:.2f}L)')

In [ ]:
# Polynomial regression (degree 2) as a third method
X     = india[india.year_start <= TRAIN_CUTOFF]['year_start'].values.reshape(-1,1)
y     = train_ts.values
poly  = PolynomialFeatures(degree=2)
Xp    = poly.fit_transform(X)
reg   = LinearRegression().fit(Xp, y)
r2    = reg.score(Xp, y)

X_fc  = np.array([india.year_start.max() + 1, india.year_start.max() + 2]).reshape(-1,1)
poly_fc = reg.predict(poly.transform(X_fc))

print(f'Polynomial (deg 2) R2 on train: {r2:.4f}')
for lbl, val in zip(fc_labels, poly_fc):
    print(f'  Forecast FY{lbl}: {val:,.0f} units ({val/100000:.2f}L)')

## 7. Model Comparison and Ensemble Forecast

In [ ]:
# Ensemble: weighted average (HW weight 0.45, ARIMA 0.35, Poly 0.20)
W_HW, W_AR, W_PO = 0.45, 0.35, 0.20
ensemble_fc = []
for hw, ar, po in zip(hw_fc_fwd, arima_fc_fwd, poly_fc):
    ensemble_fc.append(W_HW * hw + W_AR * ar + W_PO * po)

# Summary table
comparison = pd.DataFrame({
    'Fiscal Year':               fc_labels,
    'Holt-Winters':              [f'{v:,.0f}' for v in hw_fc_fwd],
    'ARIMA':                     [f'{v:,.0f}' for v in arima_fc_fwd],
    'Polynomial Reg':            [f'{v:,.0f}' for v in poly_fc],
    'ENSEMBLE (Wtd Avg)':        [f'{v:,.0f}' for v in ensemble_fc],
    'Ensemble (Lakh)':           [f'{v/100000:.2f}L' for v in ensemble_fc],
    'Lower (−8%)':               [f'{v*0.92/100000:.2f}L' for v in ensemble_fc],
    'Upper (+8%)':               [f'{v*1.08/100000:.2f}L' for v in ensemble_fc],
})

print('\n=== FORECAST COMPARISON ===')
print(comparison.to_string(index=False))

accuracy = pd.DataFrame({
    'Model':         ['Holt-Winters','ARIMA','Polynomial Reg'],
    'MAPE (test)':   [f'{mape_hw:.2%}', f'{mape_arima:.2%}', 'N/A (no test eval)'],
    'Weight':        [W_HW, W_AR, W_PO],
})
print('\n=== MODEL ACCURACY ===')
print(accuracy.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: Full forecast chart ─────────────────────────────────────────────
ax = axes[0]
ax.fill_between(india.fiscal_year, india.units_lakh, alpha=0.1, color=GREEN)
ax.plot(india.fiscal_year, india.units_lakh, color=GREEN, lw=2, label='Actual', marker='o', ms=4)

fc_x = fc_labels
fc_y = [v/100000 for v in ensemble_fc]
up   = [v*1.08/100000 for v in ensemble_fc]
lo   = [v*0.92/100000 for v in ensemble_fc]

ax.plot(fc_x, fc_y, color=AMBER, lw=2.5, ls='--', label='Ensemble Forecast', marker='D', ms=8)
ax.fill_between(fc_x, lo, up, alpha=0.2, color=AMBER, label='Confidence Band +/-8%')

ax.plot(fc_x, [v/100000 for v in hw_fc_fwd],  color=BLUE,   ls=':', lw=1.5, label='Holt-Winters')
ax.plot(fc_x, [v/100000 for v in arima_fc_fwd], color=PURPLE, ls=':', lw=1.5, label='ARIMA')
ax.plot(fc_x, [v/100000 for v in poly_fc],    color=GRAY,   ls=':', lw=1.5, label='Poly Reg')

ax.set_title('India Tractor Sales Forecast FY2025-27', fontsize=13)
ax.set_ylabel('Units (Lakh)')
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=8)
ax.grid(alpha=0.15)

# ── Right: HP Segment Forecast ────────────────────────────────────────────
ax2 = axes[1]
seg_names  = df_hp_fc['segment'].tolist()
x_pos      = np.arange(len(seg_names))
w          = 0.3
base_units = hp_data.iloc[-1][segs].tolist()
fy26       = df_hp_fc['fy2526_units'].tolist()
fy27       = df_hp_fc['fy2627_units'].tolist()

ax2.bar(x_pos - w, [b/1000 for b in base_units], w, label='FY2024-25 Actual', color=GRAY)
ax2.bar(x_pos,     [f/1000 for f in fy26],        w, label='FY2025-26 Forecast', color=GREEN)
ax2.bar(x_pos + w, [f/1000 for f in fy27],        w, label='FY2026-27 Forecast', color=AMBER)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(['<30HP','30-40HP','41-50HP','51-60HP','>60HP'], fontsize=9)
ax2.set_title('HP Segment Forecast (DuckDB Weighted)', fontsize=13)
ax2.set_ylabel('Units (000s)')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.15, axis='y')

plt.tight_layout()
plt.savefig('forecast_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Forecast chart saved as forecast_charts.png')

## 8. Emerging High-Growth States Analysis

In [ ]:
emerging = pd.DataFrame({
    'state':             ['Madhya Pradesh','Telangana','Odisha','Chhattisgarh','Jharkhand'],
    'current_share_pct': [11.6, 3.2, 2.1, 1.8, 1.4],
    'tractors_per_kha':  [4.1, 3.2, 1.8, 1.4, 1.1],
    'national_avg_kha':  [6.4, 6.4, 6.4, 6.4, 6.4],
    'net_sown_area_mha': [16.4, 5.8, 6.3, 4.1, 1.8],
    'cagr_forecast_pct': [12.1, 9.4, 8.7, 7.9, 7.1],
})

# Potential units at national average penetration
emerging['current_units_est']  = (emerging['current_share_pct'] / 100 * 912000).astype(int)
emerging['potential_units']    = ((emerging['tractors_per_kha'] / emerging['national_avg_kha'])
                                   .apply(lambda x: 1/x if x < 1 else 1)
                                   * emerging['current_units_est'] * 1.6).astype(int)
emerging['upside_units']       = emerging['potential_units'] - emerging['current_units_est']
emerging['fy27_forecast']      = (emerging['current_units_est'] *
                                   (1 + emerging['cagr_forecast_pct']/100)**2).astype(int)

print('Emerging High-Growth States — Forecast FY2026-27:')
print(emerging[['state','current_share_pct','tractors_per_kha','cagr_forecast_pct',
                'current_units_est','fy27_forecast','upside_units']].to_string(index=False))

total_upside = emerging['upside_units'].sum()
print(f'\nTotal structural upside from 5 emerging states: ~{total_upside:,} units')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
bars = ax.barh(emerging.state, emerging.cagr_forecast_pct, color=[GREEN,AMBER,BLUE,PURPLE,RED])
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_title('Forecast CAGR FY2026-28 (%)', fontsize=12)
ax.set_xlabel('CAGR (%)')
ax.axvline(5, color='white', ls='--', lw=0.8, alpha=0.5, label='5% threshold')
ax.legend(fontsize=8)

ax2 = axes[1]
x = np.arange(len(emerging))
ax2.bar(x - 0.2, emerging.tractors_per_kha, 0.4, label='Current (T/1000ha)', color=GRAY)
ax2.bar(x + 0.2, emerging.national_avg_kha, 0.4, label='National Avg (6.4)', color=GREEN)
ax2.set_xticks(x)
ax2.set_xticklabels(emerging.state, rotation=15, ha='right', fontsize=8)
ax2.set_title('Tractor Penetration vs National Average', fontsize=12)
ax2.set_ylabel('Tractors per 1000 ha')
ax2.legend(fontsize=9)

plt.tight_layout()
plt.savefig('emerging_states.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Results Summary

In [ ]:
print('=' * 60)
print('INDIA TRACTOR MARKET — FORECAST SUMMARY')
print('=' * 60)
print()
print('TOTAL MARKET FORECAST (Ensemble Model):')
for lbl, val in zip(fc_labels, ensemble_fc):
    print(f'  FY{lbl} : {val:,.0f} units ({val/100000:.2f}L) [{val*0.92/100000:.2f}L–{val*1.08/100000:.2f}L]')

print()
print('HP SEGMENT FORECAST (DuckDB Weighted-Share):')
for _, row in df_hp_fc.iterrows():
    print(f'  {row.segment:<14}: FY26={row.fy2526_units:>7,}  FY27={row.fy2627_units:>7,}  Share={row.forecast_share_pct:.1f}%')

print()
print('EMERGING STATES (High-growth, underpenetrated):')
for _, row in emerging.iterrows():
    print(f'  {row.state:<20}: CAGR={row.cagr_forecast_pct:.1f}%  FY27={row.fy27_forecast:,}  Upside={row.upside_units:,}')

print()
print('KEY ASSUMPTIONS:')
print('  - Normal SW Monsoon (868mm+) for FY2026')
print('  - Wheat MSP hike of 6-8% in FY2026 budget')
print('  - No major NBFC/banking credit disruption')
print('  - SMAM subsidy continued at current levels')
print()
print('RISK FACTORS:')
print('  - Deficit monsoon (<868mm): -8 to -12% demand impact')
print('  - Input cost (steel) spike >15%: -5% demand impact')
print('  - EV transition uncertainty in 41-50 HP segment')
print('  - Credit tightening by rural NBFCs post-RBI scrutiny')

## 10. Export Results

Export forecast tables to CSV for use in Looker Studio / Streamlit app.

In [ ]:
# Total forecast
fc_export = pd.DataFrame({
    'fiscal_year':    fc_labels,
    'ensemble_units': [int(v) for v in ensemble_fc],
    'hw_units':       [int(v) for v in hw_fc_fwd],
    'arima_units':    [int(v) for v in arima_fc_fwd],
    'poly_units':     [int(v) for v in poly_fc],
    'lower_bound':    [int(v*0.92) for v in ensemble_fc],
    'upper_bound':    [int(v*1.08) for v in ensemble_fc],
})
fc_export.to_csv('forecast_total.csv', index=False)

# HP segment forecast
df_hp_fc.to_csv('forecast_hp_segments.csv', index=False)

# Emerging states
emerging.to_csv('forecast_emerging_states.csv', index=False)

print('Exported:')
print('  forecast_total.csv')
print('  forecast_hp_segments.csv')
print('  forecast_emerging_states.csv')
print('  eda_summary.png')
print('  forecast_charts.png')
print('  emerging_states.png')
print()
print('Upload forecast_total.csv and forecast_hp_segments.csv to Google Sheets')
print('to connect them to Looker Studio as additional data sources.')